In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
%cd drive/MyDrive/project/semantic-segmentation-roads/eomt

/content/drive/MyDrive/project/semantic-segmentation-roads/eomt


In [ ]:
!pip install -r requirements.txt

In [ ]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib
from torchmetrics.classification import MulticlassJaccardIndex
from tqdm.notebook import tqdm

seed_everything(0, verbose=False)

device = 0
LIMIT_BATCHES = None
# 1. Configurazione del modello (COCO)
config_coco_path = "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
# 2. Configurazione del dataset (Cityscapes)
config_cityscapes_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

data_path = "/content/drive/MyDrive/project/"

state_dict_path_coco = "/content/drive/MyDrive/project/eomt_coco.bin"
state_dict_path_cityscapes = "/content/drive/MyDrive/project/eomt_cityscapes.bin"


with open(config_coco_path, "r") as f:
    config_coco = yaml.safe_load(f)

with open(config_cityscapes_path, "r") as f:
    config_cityscapes = yaml.safe_load(f)


# Funzioni comuni (pipeline condivisa)

In [ ]:
IGNORE_INDEX = 19   # pixels not covered by any GT mask, and COCO classes not mapped to Cityscapes

def infer_semantic(model, img, target, device):
    """
        Returns:
        pred   (H, W) int  -- class indices in model's native space
        gt     (H, W) int  -- Cityscapes train_id [0-18] or IGNORE_INDEX
    """
    with torch.no_grad(), autocast(dtype=torch.float16, device_type='cuda'):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode='bilinear'
        )
        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        pred_array = logits[0].argmax(0).cpu().numpy()

    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].cpu().numpy()
    return pred_array, target_array


def load_weights(model, state_dict_path, device):
    state_dict = torch.load(state_dict_path, map_location=f'cuda:{device}', weights_only=True)
    model.load_state_dict(state_dict, strict=False)
    return model



def make_metric(device):
    return MulticlassJaccardIndex(
        num_classes=20,
        ignore_index=IGNORE_INDEX,
        average=None,          # returns per-class IoU
        validate_args=False,
    ).to(f'cuda:{device}')


# Cityscapes Validation Dataset

In [ ]:
data_module_name, class_name = config_cityscapes["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config_cityscapes["data"].get("init_args", {})



data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
).setup()

val_loader  = data.val_dataloader()

#DEBUG
val_dataset = data.cityscapes_val_dataset
print(f'Val set size: {len(val_dataset)} images')

Val set size: 500 images


## Modello A (Cityscapes)

In [ ]:
# Load encoder
encoder_cfg = config_cityscapes["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=data.img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = config_cityscapes["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=data.num_classes,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = config_cityscapes["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in config_cityscapes["model"]["init_args"].items() if k != "network"}

model_cityscapes = (
    lit_cls(
        img_size=data.img_size,
        num_classes=data.num_classes,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


In [ ]:
model_cityscapes = load_weights(model_cityscapes, state_dict_path_cityscapes, device)

In [ ]:
metric_cityscapes = make_metric(device)
total = LIMIT_BATCHES if LIMIT_BATCHES else len(val_loader)
print(f'[Model A] Evaluating on {total} images...')

for batch_idx, batch in enumerate(tqdm(val_loader, total=total, desc='Model A - Cityscapes')):
    if LIMIT_BATCHES and batch_idx >= LIMIT_BATCHES:
        break

    imgs, targets = batch
    for img, target in zip(imgs, targets):
      pred, gt = infer_semantic(model_cityscapes, img, target, device)

      # GT is already in [0..18] ∪ {IGNORE_INDEX}.
      metric_cityscapes.update(
          torch.from_numpy(pred.astype(np.int64)).to(f'cuda:{device}'),
          torch.from_numpy(gt.astype(np.int64)).to(f'cuda:{device}')
      )

iou_per_class_cs = metric_cityscapes.compute()[:19].cpu().numpy()  # only first 19 classes
miou_cityscapes = float(iou_per_class_cs.mean())
print(f'\n[Model A] mIoU (Cityscapes Semantic): {miou_cityscapes * 100:.2f}%')

[Model A] Evaluating on 500 images...


Model A - Cityscapes:   0%|          | 0/500 [00:00<?, ?it/s]


[Model A] mIoU (Cityscapes Semantic): 81.68%


## Modello (COCO)

In [ ]:
encoder_cfg = config_coco["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)


coco_data_init_args = config_coco["data"].get("init_args", {})
model_img_size = coco_data_init_args.get("img_size", (640, 640))
model_num_classes = coco_data_init_args.get("num_classes", 133)
encoder = encoder_cls(img_size=model_img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = config_coco["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}

network = network_cls(
    masked_attn_enabled=False,
    num_classes=model_num_classes,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = config_coco["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
#model_kwargs = {k: v for k, v in config_model["model"]["init_args"].items() if k != "network"}

#if "stuff_classes" in coco_data_init_args:
#   model_kwargs["stuff_classes"] = config_model["data"]["init_args"]["stuff_classes"]

model_kwargs["stuff_classes"] = coco_data_init_args.get("stuff_classes", list(range(80, 133)))

model_coco = (
    lit_cls(
        img_size=model_img_size,
        num_classes=model_num_classes,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

In [ ]:
# Inizializziamo a 19 (Ignore Index sicuro per torchmetrics)
coco_to_cityscapes = np.full(256, 19, dtype=np.uint8)

# Mappatura manuale derivata dalla ricerca sulle categorie
coco_to_cityscapes[0] = 11   # person -> person
coco_to_cityscapes[1] = 18   # bicycle -> bicycle
coco_to_cityscapes[2] = 13   # car -> car
coco_to_cityscapes[3] = 17   # motorcycle -> motorcycle
coco_to_cityscapes[5] = 15   # bus -> bus
coco_to_cityscapes[6] = 16   # train -> train
coco_to_cityscapes[7] = 14   # truck -> truck
coco_to_cityscapes[9] = 6    # traffic light -> traffic light
coco_to_cityscapes[11] = 7   # stop sign -> traffic sign
coco_to_cityscapes[82] = 2   # bridge -> building
coco_to_cityscapes[90] = 9   # gravel -> terrain
coco_to_cityscapes[91] = 2   # house -> building
coco_to_cityscapes[100] = 0  # road -> road
coco_to_cityscapes[101] = 2  # roof -> building
coco_to_cityscapes[102] = 9  # sand -> terrain
coco_to_cityscapes[109] = 3  # wall-brick -> wall
coco_to_cityscapes[110] = 3  # wall-stone -> wall
coco_to_cityscapes[111] = 3  # wall-tile -> wall
coco_to_cityscapes[112] = 3  # wall-wood -> wall
coco_to_cityscapes[116] = 8  # tree-merged -> vegetation
coco_to_cityscapes[117] = 4  # fence-merged -> fence
coco_to_cityscapes[119] = 10 # sky-other-merged -> sky
coco_to_cityscapes[123] = 1  # pavement-merged -> sidewalk
coco_to_cityscapes[125] = 8  # grass-merged -> vegetation
coco_to_cityscapes[126] = 9  # dirt-merged -> terrain
coco_to_cityscapes[129] = 2  # building-other-merged -> building
coco_to_cityscapes[131] = 3  # wall-other-merged -> wall

In [ ]:
model_coco = load_weights(model_coco, state_dict_path_coco, device)

In [ ]:
metric_coco = make_metric(device)
total = LIMIT_BATCHES if LIMIT_BATCHES else len(val_loader)
print(f'[Model B] Evaluating on {total} images...')

for batch_idx, batch in enumerate(tqdm(val_loader, total=total, desc='Model B - COCO')):
    if LIMIT_BATCHES and batch_idx >= LIMIT_BATCHES:
        break

    imgs, targets = batch
    for img, target in zip(imgs, targets):
      pred_coco, gt = infer_semantic(model_coco, img, target, device)


      pred_mapped = coco_to_cityscapes[pred_coco]

      metric_coco.update(
          torch.from_numpy(pred_mapped.astype(np.int64)).to(f'cuda:{device}'),
          torch.from_numpy(gt.astype(np.int64)).to(f'cuda:{device}')
      )

iou_per_class_coco = metric_coco.compute()[:19].cpu().numpy()
miou_coco = float(iou_per_class_coco.mean())
print(f'\n[Model B] mIoU (COCO → Cityscapes): {miou_coco * 100:.2f}%')

[Model B] Evaluating on 500 images...


Model B - COCO:   0%|          | 0/500 [00:00<?, ?it/s]


[Model B] mIoU (COCO → Cityscapes): 51.79%


# Riassunto Statistiche

In [ ]:
# Cityscapes class names (train_id 0..18)
CS_CLASS_NAMES = [
    'road', 'sidewalk', 'building', 'wall', 'fence',
    'pole', 'traffic light', 'traffic sign', 'vegetation', 'terrain',
    'sky', 'person', 'rider', 'car', 'truck',
    'bus', 'train', 'motorcycle', 'bicycle'
]


print('=' * 60)
print('         SEMANTIC SEGMENTATION EVALUATION SUMMARY')
print('         Dataset: Cityscapes val (500 images)')
print('         Metric : mIoU over 19 Cityscapes classes')
print('=' * 60)
print(f'  Model  [Cityscapes Semantic]  :  {miou_cityscapes * 100:.2f}%')
print(f'  Model  [COCO Panoptic → Sem.] :  {miou_coco * 100:.2f}%')
print('=' * 60)

print('\nPer-class IoU:')
print(f'{"Class":<18} {"Model (Cityscapes)":>14} {"Model (COCO)":>16}')
print('-' * 52)
for i, name in enumerate(CS_CLASS_NAMES):
    iou_cityscapes = iou_per_class_cs[i]   * 100
    iou_coco = iou_per_class_coco[i] * 100
    print(f'{name:<18} {iou_cityscapes:>12.1f}%  {iou_coco:>13.1f}%')
print('-' * 52)
print(f'{"mIoU":<18} {miou_cityscapes*100:>12.1f}%  {miou_coco*100:>13.1f}%')

         SEMANTIC SEGMENTATION EVALUATION SUMMARY
         Dataset: Cityscapes val (500 images)
         Metric : mIoU over 19 Cityscapes classes
  Model  [Cityscapes Semantic]  :  81.68%
  Model  [COCO Panoptic → Sem.] :  45.94%

Per-class IoU:
Class              Model (Cityscapes)     Model (COCO)
----------------------------------------------------
road                       98.4%           93.2%
sidewalk                   87.4%           61.2%
building                   94.1%           81.1%
wall                       66.1%           46.9%
fence                      65.5%           37.6%
pole                       71.0%            0.0%
traffic light              75.0%           41.3%
traffic sign               82.1%            1.3%
vegetation                 93.0%           83.0%
terrain                    66.6%            2.0%
sky                        95.5%           88.3%
person                     85.4%           58.4%
rider                      71.1%            0.0%
car      